# 00 — Modelagem de Dados: Pokémon Lakehouse

Este notebook documenta **a fonte de dados** e o **modelo dimensional** usado nos notebooks seguintes (Delta Lake e Iceberg).

## Fonte de dados

- **Origem**: [Pokemon — Kaggle (abcsds/pokemon)](https://www.kaggle.com/datasets/abcsds/pokemon)
- **Mirror público usado** (sem login Kaggle): [GitHub Gist — armgilles/pokemon.csv](https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv)
- **Volume**: 800 Pokémons, gerações 1 a 6.
- **Tamanho**: ~45 KB (CSV).
- **Licença**: dados públicos (atributos de jogo, sem PII).

### Schema bruto (CSV)

| Coluna | Tipo | Descrição |
|---|---|---|
| `#` | int | ID Pokédex |
| `Name` | string | Nome do Pokémon |
| `Type 1` | string | Tipo principal (Fire, Water, Grass...) |
| `Type 2` | string \| null | Tipo secundário (opcional) |
| `Total` | int | Soma das stats |
| `HP`, `Attack`, `Defense`, `Sp. Atk`, `Sp. Def`, `Speed` | int | Atributos de combate |
| `Generation` | int (1..6) | Geração |
| `Legendary` | bool | É lendário |

## Modelo dimensional (estrela)

O CSV é uma **única tabela desnormalizada**. Para demonstrar JOINs, schema evolution e partition evolution nos notebooks Delta/Iceberg, normalizamos em **modelo estrela** com 3 tabelas:

```mermaid
erDiagram
    DIM_TYPE ||--o{ FACT_POKEMON : type_1_id
    DIM_TYPE ||--o{ FACT_POKEMON : type_2_id
    DIM_GENERATION ||--o{ FACT_POKEMON : generation_id

    DIM_TYPE {
        int    type_id PK
        string type_name
    }
    DIM_GENERATION {
        int    generation_id PK
        string generation_name
        string era
    }
    FACT_POKEMON {
        int     pokemon_id PK
        string  name
        int     type_1_id FK
        int     type_2_id FK
        int     generation_id FK
        int     hp
        int     attack
        int     defense
        int     sp_atk
        int     sp_def
        int     speed
        int     total
        boolean is_legendary
    }
```

**Decisões de modelagem:**
- `dim_type` é **conformada** (mesma dimensão usada por `type_1_id` e `type_2_id`).
- `fact_pokemon` será **particionada por `generation_id`** nos notebooks de Delta e Iceberg (boa cardinalidade: 6 partições, ~133 linhas cada).
- `total` é coluna calculada do CSV original — mantemos para fidelidade à fonte e para usar em filtros de DELETE.
- Nomes em snake_case e sem espaços para evitar problemas com SQL.

## DDL — versão Delta SQL

```sql
-- dim_type
CREATE TABLE IF NOT EXISTS dim_type (
    type_id   INT  NOT NULL,
    type_name STRING NOT NULL
) USING DELTA
LOCATION '/workspace/data/delta/dim_type';

-- dim_generation
CREATE TABLE IF NOT EXISTS dim_generation (
    generation_id   INT  NOT NULL,
    generation_name STRING NOT NULL,
    era             STRING
) USING DELTA
LOCATION '/workspace/data/delta/dim_generation';

-- fact_pokemon (particionada por generation_id)
CREATE TABLE IF NOT EXISTS fact_pokemon (
    pokemon_id   INT NOT NULL,
    name         STRING NOT NULL,
    type_1_id    INT NOT NULL,
    type_2_id    INT,
    hp           INT, attack  INT, defense INT,
    sp_atk       INT, sp_def  INT, speed   INT,
    total        INT,
    is_legendary BOOLEAN,
    generation_id INT NOT NULL
) USING DELTA
PARTITIONED BY (generation_id)
LOCATION '/workspace/data/delta/fact_pokemon';
```

## DDL — versão Iceberg SQL

```sql
CREATE TABLE local.pokedex.dim_type (
    type_id   INT,
    type_name STRING
) USING iceberg;

CREATE TABLE local.pokedex.dim_generation (
    generation_id   INT,
    generation_name STRING,
    era             STRING
) USING iceberg;

CREATE TABLE local.pokedex.fact_pokemon (
    pokemon_id   INT,
    name         STRING,
    type_1_id    INT,
    type_2_id    INT,
    hp           INT, attack  INT, defense INT,
    sp_atk       INT, sp_def  INT, speed   INT,
    total        INT,
    is_legendary BOOLEAN,
    generation_id INT
) USING iceberg
PARTITIONED BY (generation_id);
```

## Pipeline bronze → silver → gold

Vamos executar o pipeline completo aqui para deixar dataframes prontos em **Parquet** em `/workspace/data/gold/`. Os notebooks Delta e Iceberg leem desse local.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType

spark = (
    SparkSession.builder
    .appName("PokemonModeling")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark.version

In [ ]:
# Bronze — CSV cru
RAW_PATH = "/workspace/data/raw/Pokemon.csv"

bronze = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RAW_PATH)
)
print(f"Bronze rows: {bronze.count()}")
bronze.printSchema()
bronze.show(5, truncate=False)

In [ ]:
# Silver — limpeza: renomear colunas (sem espaços/pontuação), tipar, tratar nulls
silver = (
    bronze
    .withColumnRenamed("#", "pokemon_id")
    .withColumnRenamed("Name", "name")
    .withColumnRenamed("Type 1", "type_1")
    .withColumnRenamed("Type 2", "type_2")
    .withColumnRenamed("Total", "total")
    .withColumnRenamed("HP", "hp")
    .withColumnRenamed("Attack", "attack")
    .withColumnRenamed("Defense", "defense")
    .withColumnRenamed("Sp. Atk", "sp_atk")
    .withColumnRenamed("Sp. Def", "sp_def")
    .withColumnRenamed("Speed", "speed")
    .withColumnRenamed("Generation", "generation_id")
    .withColumnRenamed("Legendary", "is_legendary")
    # type_2 vem como string vazia em algumas linhas — converter para NULL
    .withColumn(
        "type_2",
        F.when(F.col("type_2").isNull() | (F.trim(F.col("type_2")) == ""), None)
         .otherwise(F.col("type_2")),
    )
    .withColumn("is_legendary", F.col("is_legendary").cast(BooleanType()))
)
silver.printSchema()
silver.show(5, truncate=False)

In [ ]:
# Gold — explodir em modelo estrela

# 1. dim_type (conformada — vem de type_1 e type_2)
types_1 = silver.select(F.col("type_1").alias("type_name"))
types_2 = silver.select(F.col("type_2").alias("type_name")).where(F.col("type_name").isNotNull())

dim_type = (
    types_1.union(types_2)
    .distinct()
    .orderBy("type_name")
    .withColumn("type_id", F.row_number().over(Window.orderBy("type_name")))
    .select("type_id", "type_name")
)
print("dim_type:")
dim_type.show(truncate=False)

In [ ]:
# 2. dim_generation
dim_generation = spark.createDataFrame(
    [
        (1, "Kanto",   "Classic"),
        (2, "Johto",   "Classic"),
        (3, "Hoenn",   "Advanced"),
        (4, "Sinnoh",  "Advanced"),
        (5, "Unova",   "Modern"),
        (6, "Kalos",   "Modern"),
    ],
    schema=StructType([
        StructField("generation_id", IntegerType(), False),
        StructField("generation_name", StringType(), False),
        StructField("era", StringType(), True),
    ]),
)
dim_generation.show()

In [ ]:
# 3. fact_pokemon — substitui type_1/type_2 por type_1_id/type_2_id
fact_pokemon = (
    silver.alias("s")
    .join(dim_type.alias("t1"), F.col("s.type_1") == F.col("t1.type_name"), "left")
    .join(dim_type.alias("t2"), F.col("s.type_2") == F.col("t2.type_name"), "left")
    .select(
        F.col("s.pokemon_id"),
        F.col("s.name"),
        F.col("t1.type_id").alias("type_1_id"),
        F.col("t2.type_id").alias("type_2_id"),
        F.col("s.hp"), F.col("s.attack"), F.col("s.defense"),
        F.col("s.sp_atk"), F.col("s.sp_def"), F.col("s.speed"),
        F.col("s.total"),
        F.col("s.is_legendary"),
        F.col("s.generation_id"),
    )
)
fact_pokemon.printSchema()
fact_pokemon.show(5, truncate=False)
print(f"fact_pokemon rows: {fact_pokemon.count()}")

In [ ]:
# Persistir gold em Parquet — fonte para os notebooks Delta e Iceberg
GOLD_DIR = "/workspace/data/gold"

(dim_type.coalesce(1)
    .write.mode("overwrite").parquet(f"{GOLD_DIR}/dim_type"))
(dim_generation.coalesce(1)
    .write.mode("overwrite").parquet(f"{GOLD_DIR}/dim_generation"))
(fact_pokemon.coalesce(1)
    .write.mode("overwrite").parquet(f"{GOLD_DIR}/fact_pokemon"))

print(f"Gold escrito em {GOLD_DIR}/")
import os
for d in os.listdir(GOLD_DIR):
    print(" -", d)

In [ ]:
spark.stop()

## Próximos passos

- **`01_delta_lake.ipynb`** — escreve o gold em **Delta Lake** e demonstra INSERT, UPDATE, DELETE, MERGE, schema evolution, time travel, OPTIMIZE e VACUUM.
- **`02_apache_iceberg.ipynb`** — escreve o gold em **Apache Iceberg** e demonstra as mesmas operações + **partition evolution** (recurso exclusivo do Iceberg).